# 01 — Zero-fit arms on `L_ctrl`

Cached exp_2 predictions (`exp1_ref`, `exp2_aug`, and the 3-seed ensemble `exp3_ref_ens3`) are re-selected (temperature + decoder, nested) and scored against `L_ctrl` on both views. The rule arms (baseline, rasya C06/C08) and ryu's shipped pipeline are re-run and scored against every label set. Nothing is fitted.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp3_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_3')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp3_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp3_protocol as P

PREREG verified cff444fff0106e9c | frozen 2026-11-08T16:53:50+00:00


In [2]:
cache = {}
for variant in ('exp3_ref_ens3', 'exp2_aug', 'exp1_ref'):
    for view in C.VIEWS:
        t0 = time.time()
        result = P.evaluate_variant(variant, view, C.EVAL_LABEL, cache)
        C1.write_json(C.RESULTS / f'variant_{variant}__{view}__{C.EVAL_LABEL}.json', result)
        frame = pd.DataFrame(result['rows'])
        print(f'== {variant:14s} {view:12s} L_ctrl mean spellF1 {frame.spell_f1.mean():.4f}  max|count err| {frame.count_error_ratio.abs().max():.3f}  ({time.time() - t0:.0f}s)')
        print('   final config:', {k: round(v, 3) for k, v in result['final_config'].items()})

== exp3_ref_ens3  original     L_ctrl mean spellF1 0.7506  max|count err| 0.195  (79s)
   final config: {'temperature': 1.181, 'lambda_team': 8.0, 'lambda_dead': 4.0, 'gamma': 0.0}


== exp3_ref_ens3  target_like  L_ctrl mean spellF1 0.7483  max|count err| 0.225  (80s)
   final config: {'temperature': 1.17, 'lambda_team': 8.0, 'lambda_dead': 4.0, 'gamma': 0.0}


== exp2_aug       original     L_ctrl mean spellF1 0.7298  max|count err| 0.217  (87s)
   final config: {'temperature': 1.315, 'lambda_team': 8.0, 'lambda_dead': 4.0, 'gamma': 0.0}


== exp2_aug       target_like  L_ctrl mean spellF1 0.7346  max|count err| 0.221  (89s)
   final config: {'temperature': 1.297, 'lambda_team': 8.0, 'lambda_dead': 4.0, 'gamma': 0.0}


== exp1_ref       original     L_ctrl mean spellF1 0.7354  max|count err| 0.176  (94s)
   final config: {'temperature': 1.315, 'lambda_team': 8.0, 'lambda_dead': 2.0, 'gamma': 0.5}


== exp1_ref       target_like  L_ctrl mean spellF1 0.7291  max|count err| 0.190  (91s)
   final config: {'temperature': 1.306, 'lambda_team': 8.0, 'lambda_dead': 4.0, 'gamma': 0.0}


In [3]:
t0 = time.time()
rows = []
for view in C.VIEWS:
    rows += P.score_reference(P.reference_predictions(view), view, ('ctrl', 'provider', 'track_1', 'track_3'))
reference = pd.DataFrame(rows)
reference.to_csv(C.RESULTS / 'reference_rows.csv', index=False)
print(reference.groupby(['label_set', 'view', 'variant'])[['spell_f1', 'macro_f1', 'count_error_ratio']].mean().round(4).to_string())
print(f'{time.time() - t0:.0f}s')

                                 spell_f1  macro_f1  count_error_ratio
label_set view        variant                                         
ctrl      original    baseline     0.1985    0.4588             0.7764
                      rasya_c06    0.3634    0.6353            -0.0333
                      rasya_c08    0.3649    0.5925            -0.1133
                      ryu_final    0.6287    0.8869            -0.1825
          target_like baseline     0.2190    0.6151             0.8568
                      rasya_c06    0.2964    0.4035            -0.2483
                      rasya_c08    0.3043    0.3746            -0.2858
                      ryu_final    0.6192    0.8784            -0.1664
provider  original    baseline     0.1861    0.4579             0.1790
                      rasya_c06    0.3223    0.6342            -0.3571
                      rasya_c08    0.3249    0.5915            -0.4107
                      ryu_final    0.4874    0.8787            -0.4570
      